In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", None)

### Load Clean Dataset

In [ ]:
path = "jobs_clean_for_model_v2.csv"
df = pd.read_csv(path)

print("Loaded shape:", df.shape)
df.head(5)

Loaded shape: (4704, 6)


,text_all,company,location_restrictions,employment_type,seniority,salary_target
0,"Sr. Business Analyst, Go-to-Market |",Crowdstrike,United States,Full Time,Senior,152500.0
1,"Director, Product Marketing |",Nox Medical,United States,Full Time,Director,145000.0
2,Barry County Assistance Payments Worker 8-11 (13) |,State of Michigan,United States,Full Time,Entry-level,59591.5
3,Technical Support Engineer II - HL7 | Developer,GE HealthCare,United States,Full Time,Mid-level,135000.0
4,PubSec Project Admin |,SHI International Corp.,United States,Part Time,Entry-level,41600.0


### Basic Checks

In [ ]:
# Quick sanity checks before training
print(df.dtypes)
print("\nMissing ratios:")
display(df.isna().mean().sort_values(ascending=False).head(10))

display(df["salary_target"].describe())

text_all                  object
company                   object
location_restrictions     object
employment_type           object
seniority                 object
salary_target            float64
dtype: object

Missing ratios:


location_restrictions    0.009991
text_all                 0.000000
company                  0.000000
employment_type          0.000000
seniority                0.000000
salary_target            0.000000
dtype: float64

count      4704.000000
mean     109168.796237
std       72274.668071
min       15000.000000
25%       52500.000000
50%       95680.000000
75%      145500.000000
max      600000.000000
Name: salary_target, dtype: float64

### Define Features (X) and Target (y)

In [ ]:
feature_cols = [
    "text_all",
    "company",
    "location_restrictions",
    "employment_type",
    "seniority",
]
target_col = "salary_target"

X = df[feature_cols].copy()
y = df[target_col].astype(float).copy()

print("X columns:", list(X.columns))

X columns: ['text_all', 'company', 'location_restrictions', 'employment_type', 'seniority']


### Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (3763, 5) Test: (941, 5)


### Build Preprocessing + Model Pipeline

In [ ]:
# Column groups
text_col = "text_all"
cat_cols = ["company", "location_restrictions", "employment_type", "seniority"]

# Preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            max_features=50000
        ), text_col),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ],
    remainder="drop"
)

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

# preprocess -> model
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("rf", rf)
])

pipe

,steps,"[('preprocess', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### GridSearchCV

In [ ]:
# hyperparameter grid
param_grid = {
    "rf__n_estimators": [300, 600],
    "rf__max_depth": [None, 20, 40],
    "rf__min_samples_split": [2, 5],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", 0.5],
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("Best params:")
print(grid.best_params_)
print("Best CV MAE:", -grid.best_score_)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=5, rf__n_estimators=300; total time=   9.4s
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=5, rf__n_estimators=300; total time=   9.7s
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=300; total time=  17.2s
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=300; total time=  17.4s
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=2, rf__n_estimators=300; total time=  17.5s
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=1, rf__min_samples_split=5, rf__n_estimators=300; total time=   8.1s
[CV] END rf__max_depth=None, rf__max_features=sqrt, rf__min_samples_leaf=2, rf__min_sa

### Evaluate on Test Set

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred) 
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Test MAE :", mae)
print("Test RMSE:", rmse)
print("Test R2  :", r2)

Test MAE : 25332.631773822177
Test RMSE: 41390.268342150295
Test R2  : 0.6326543423620243


In [31]:
import joblib

joblib.dump(best_model, "salary_model_rf.joblib")
print("Saved: salary_model_rf.joblib")

Saved: salary_model_rf.joblib
